# Higgs uncertainty 

In [1]:
import numpy as np
import pandas as pd
import os
import h5py
import importlib
import tensorflow as tf
import pickle
import yaml
import common.datasets_hephy as datasets_hephy
from data_loader.data_loader_2 import H5DataLoader

2025-04-29 19:39:26.568853: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-04-29 19:39:26.676151: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
model_dir = "./demodir/models"
processes = ['htautau','ztautau','ttbar','diboson']

## Prepare the data

In [3]:
### FIXME: this part should download the data file from Zenodo 
#training_data_dir = "/scratch-cbe/users/robert.schoefbeck/Higgs_uncertainty/data/"
training_data_dir = "/home/daohan/apps/Higgs"

selection = "lowMT_VBFJet"

## Split the combined dataset

In [39]:
input_file = "/home/daohan/apps/Higgs/lowMT_VBFJet/jes_1p01_met_1.h5"
output_dir = "/home/daohan/apps/Higgs/lowMT_VBFJet"
input_basename = os.path.splitext(os.path.basename(input_file))[0]

label_to_prefix = {
    0: "htautau",
    1: "ztautau",
    2: "ttbar",
    3: "diboson",
}

with h5py.File(input_file, "r") as f:
    data = f["data"][:] 
    print(f"Loaded data with shape {data.shape}")

labels = data[:, -1]

for label_value, prefix in label_to_prefix.items():
    mask = labels == label_value
    selected_data = data[mask]

    output_filename = f"{prefix}_{input_basename}.h5"
    output_path = os.path.join(output_dir, output_filename)

    with h5py.File(output_path, "w") as f_out:
        f_out.create_dataset("data", data=selected_data, compression="gzip")
    print(f"Written {selected_data.shape[0]} entries to {output_path}")

Loaded data with shape (17523787, 30)
Written 11719573 entries to /home/daohan/apps/Higgs/lowMT_VBFJet/htautau_jes_1p01_met_1.h5
Written 4219339 entries to /home/daohan/apps/Higgs/lowMT_VBFJet/ztautau_jes_1p01_met_1.h5
Written 1552923 entries to /home/daohan/apps/Higgs/lowMT_VBFJet/ttbar_jes_1p01_met_1.h5
Written 31952 entries to /home/daohan/apps/Higgs/lowMT_VBFJet/diboson_jes_1p01_met_1.h5


In [4]:
dl = datasets_hephy.get_data_loader(data_directory=training_data_dir, process=None, selection=selection)

## Train the models

Training the models takes a long time. As a proof of concept, the examples below takes a small fraction of events and train the model for 1 epoch. 

### Inclusive Crosssection

In [5]:
from ML.IC.IC import InclusiveCrosssection

In [6]:
ic_name = f"IC_{selection}"
ic_model_directory = os.path.join( model_dir, "IC" )
os.makedirs(ic_model_directory, exist_ok=True)
filename = os.path.join(ic_model_directory, ic_name)+'.pkl'
print ("Training.")
ic = InclusiveCrosssection()

ic.load_training_data(datasets_hephy, training_data_dir, selection)
#ic.train             (datasets_hephy, mva_selections.selections[args.mvaSelection] if args.mvaSelection is not None else None, small=True)
ic.train             (datasets_hephy, None, small=True)

ic.save(filename)
print ("Written %s"%( filename ))

print(f"Trained IC for this selection: {selection}")
print(ic)

Training.


Computing weight sums:   0%|                                                                                                                                                     | 0/10 [00:00<?, ?batch/s]

Written ./demodir/models/IC/IC_lowMT_VBFJet.pkl
Trained IC for this selection: lowMT_VBFJet
IC: lowMT_VBFJet                           S/B = 0.003971  yield: htautau:    22.60 ztautau:  4120.57 ttbar:  1539.10 diboson:    32.14
                                                           count: htautau:  1158182 ztautau:   412057 ttbar:   153883 diboson:     3213


### Scalar

In [7]:
from ML.Scaler.Scaler import Scaler
scalar_model_directory = os.path.join(model_dir, "Scaler")
os.makedirs(scalar_model_directory, exist_ok=True)
process_to_train = [None]+processes
for p in process_to_train:
    subdirs = [arg for arg in [p, selection] if arg is not None]
    scaler_name = "Scaler_"+"_".join(subdirs)
    filename = os.path.join(scalar_model_directory, scaler_name) + '.pkl'
    print("Training.")
    scaler = Scaler()

    scaler.load_training_data(training_data_dir=training_data_dir, datasets_hephy=datasets_hephy, selection=selection, process=p)
    scaler.train(small=True)

    scaler.save(filename)
    print(f"Written {filename}")
    

Training.


Computing feature mean/variance:   0%|                                                                                                                                           | 0/10 [00:00<?, ?batch/s]


Written ./demodir/models/Scaler/Scaler_lowMT_VBFJet.pkl
Training.


Computing feature mean/variance:   0%|                                                                                                                                           | 0/10 [00:01<?, ?batch/s]


Written ./demodir/models/Scaler/Scaler_htautau_lowMT_VBFJet.pkl
Training.


...

Written ./demodir/models/Scaler/Scaler_ztautau_lowMT_VBFJet.pkl
Training.


Computing feature mean/variance:   0%|                                                                                                                                           | 0/10 [00:00<?, ?batch/s]


Written ./demodir/models/Scaler/Scaler_ttbar_lowMT_VBFJet.pkl
Training.


Computing feature mean/variance:   0%|                                                                                                                                           | 0/10 [00:00<?, ?batch/s]

Written ./demodir/models/Scaler/Scaler_diboson_lowMT_VBFJet.pkl


### Inclusive Crosssection Parametrization

In [8]:
from ML.ICP.ICP import InclusiveCrosssectionParametrization

In [9]:
config_name = "ML.configs.icp_quad_tes_jes_met_light"
# import the training parameters in config
config = importlib.import_module("%s"%( config_name))
for p in processes:
    subdirs = [arg for arg in [p, selection, config_name.split('.')[-1]] if arg is not None]
    icp_name = "ICP_"+"_".join(subdirs)
    icp_model_directory = os.path.join( model_dir, "ICP" )
    os.makedirs(icp_model_directory, exist_ok=True)
    
    filename = os.path.join(icp_model_directory, icp_name)+'.pkl'
    

    print ("Training.")
    icp = InclusiveCrosssectionParametrization( config = config )

    icp.load_training_data(datasets_hephy=datasets_hephy, training_data_dir=training_data_dir, selection=selection, process=p) 
    icp.train             (small=True, train_ratio = True, selection=None)

    icp.save(filename)
    print ("Written %s"%( filename ))
    
    print (f"Trained ICP with config {config_name} in selection {selection}")
    prefix = "ICP: "+selection
    print (prefix.ljust(50)+icp.__str__())

Training.
ICP training data: Base point nu = (0.0, 0.0, 0.0), alpha = (1.0, 1.0, 0.0), file = /home/daohan/apps/Higgs/lowMT_VBFJet/htautau.h5
ICP training data: Base point nu = (-2.0, 0.0, 0.0), alpha = (0.98, 1.0, 0.0), file = /home/daohan/apps/Higgs/lowMT_VBFJet/htautau_tes_0p98.h5
ICP training data: Base point nu = (-1.0, -1.0, 0.0), alpha = (0.99, 0.99, 0.0), file = /home/daohan/apps/Higgs/lowMT_VBFJet/htautau_tes_0p99_jes_0p99.h5
ICP training data: Base point nu = (-1.0, 0.0, 0.0), alpha = (0.99, 1.0, 0.0), file = /home/daohan/apps/Higgs/lowMT_VBFJet/htautau_tes_0p99.h5
ICP training data: Base point nu = (-1.0, 0.0, 1.0), alpha = (0.99, 1.0, 1.0), file = /home/daohan/apps/Higgs/lowMT_VBFJet/htautau_tes_0p99_met_1.h5
ICP training data: Base point nu = (-1.0, 1.0, 0.0), alpha = (0.99, 1.01, 0.0), file = /home/daohan/apps/Higgs/lowMT_VBFJet/htautau_tes_0p99_jes_1p01.h5
ICP training data: Base point nu = (0.0, -2.0, 0.0), alpha = (1.0, 0.98, 0.0), file = /home/daohan/apps/Higgs/lowMT_

Computing weight sum:   0%|                                                                                                                                                      | 0/10 [00:01<?, ?batch/s]

Written ./demodir/models/ICP/ICP_htautau_lowMT_VBFJet_icp_quad_tes_jes_met_light.pkl
Trained ICP with config ML.configs.icp_quad_tes_jes_met_light in selection lowMT_VBFJet
ICP: lowMT_VBFJet                                 +6.0e-03*nu_tes +1.3e-02*nu_jes -2.6e-05*nu_met -1.0e-04*nu_tes*nu_tes -1.7e-04*nu_jes*nu_jes -9.2e-05*nu_met*nu_met +4.5e-05*nu_tes*nu_jes -2.3e-06*nu_tes*nu_met +2.1e-05*nu_jes*nu_met
Training.
ICP training data: Base point nu = (0.0, 0.0, 0.0), alpha = (1.0, 1.0, 0.0), file = /home/daohan/apps/Higgs/lowMT_VBFJet/ztautau.h5
ICP training data: Base point nu = (-2.0, 0.0, 0.0), alpha = (0.98, 1.0, 0.0), file = /home/daohan/apps/Higgs/lowMT_VBFJet/ztautau_tes_0p98.h5
ICP training data: Base point nu = (-1.0, -1.0, 0.0), alpha = (0.99, 0.99, 0.0), file = /home/daohan/apps/Higgs/lowMT_VBFJet/ztautau_tes_0p99_jes_0p99.h5
ICP training data: Base point nu = (-1.0, 0.0, 0.0), alpha = (0.99, 1.0, 0.0), file = /home/daohan/apps/Higgs/lowMT_VBFJet/ztautau_tes_0p99.h5
ICP train

### Multiclassifier

In [10]:
from ML.TFMC.TFMC import TFMC

Num GPUs Available:  0


In [11]:
config_name = "ML.configs.tfmc_scan_do_1"
# import the training parameters in config
config = importlib.import_module("%s"%( config_name))

# Where to store the training
TFMC_model_directory = os.path.join(model_dir, "TFMC", selection, config_name.split('.')[-1])
os.makedirs(TFMC_model_directory, exist_ok=True)

In [12]:
if config.use_ic:
    from ML.IC.IC import InclusiveCrosssection
    ic = InclusiveCrosssection.load(os.path.join(model_dir, "IC", "IC_"+selection+'.pkl'))
    config.weight_sums = ic.weight_sums
    print("We use this IC:")
    print(ic)
    
# Do we use a Scaler?
if config.use_scaler:
    from ML.Scaler.Scaler import Scaler 
    scaler = Scaler.load(os.path.join(model_dir, "Scaler", "Scaler_"+selection+'.pkl'))
    config.feature_means     = scaler.feature_means
    config.feature_variances = scaler.feature_variances

    print("We use this scaler:")
    print(scaler)

tfmc = TFMC(config)

tfmc.load_training_data(datasets_hephy, training_data_dir, selection, n_split=100)

max_batch = 1 # FIXME: -1 for full training

# Determine the starting epoch
starting_epoch = 0

# Training Loop
for epoch in range(0, config.n_epochs):

    # Manually evaluate and update the learning rate
    if hasattr(tfmc, 'lr_schedule'):  # Ensure the schedule exists
        new_lr = tfmc.lr_schedule(epoch)
        tfmc.optimizer.learning_rate.assign(new_lr)  # Update the optimizer's learning rate

    # Print the current learning rate
    current_lr = tf.keras.backend.get_value(tfmc.optimizer.learning_rate)  # Direct access
    print(f"Epoch {epoch}/{config.n_epochs} - Learning rate: {current_lr:.6f}")

    true_histograms, pred_histograms = tfmc.train_one_epoch(max_batch=max_batch, accumulate_histograms=(epoch%1==0))
    tfmc.save(TFMC_model_directory, epoch)  # Save model and config after each epoch

    #if true_histograms is not None and pred_histograms is not None:
    #    # Plot convergence
    #    tfmc.plot_convergence_root(
    #        true_histograms,
    #        pred_histograms,
    #        epoch,
    #        plot_directory,
    #        data_structure.feature_names, 
    #    )

    break

print(f"TFMC models saved in {TFMC_model_directory}")

2025-04-29 16:54:33.559004: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


We use this IC:
IC: lowMT_VBFJet                           S/B = 0.003971  yield: htautau:    22.60 ztautau:  4120.57 ttbar:  1539.10 diboson:    32.14
                                                           count: htautau:  1158182 ztautau:   412057 ttbar:   153883 diboson:     3213
We use this scaler:
Scaler: selection lowMT_VBFJet process (not set)
PRI_lep_pt: mean=46.312, variance=850.590
PRI_lep_eta: mean=-0.000, variance=1.328
PRI_lep_phi: mean=-0.002, variance=3.288
PRI_had_pt: mean=61.323, variance=1443.225
PRI_had_eta: mean=0.001, variance=1.393
PRI_had_phi: mean=0.003, variance=3.287
PRI_jet_leading_pt: mean=119.773, variance=5152.242
PRI_jet_leading_eta: mean=-0.001, variance=3.265
PRI_jet_leading_phi: mean=-0.002, variance=3.275
PRI_jet_subleading_pt: mean=61.797, variance=1126.916
PRI_jet_subleading_eta: mean=0.002, variance=4.463
PRI_jet_subleading_phi: mean=-0.001, variance=3.246
PRI_n_jets: mean=2.895, variance=1.162
PRI_jet_all_pt: mean=216.515, variance=14143.631
P

Processing Batches:   0%|                                                                                                                                                          | 0/100 [00:01<?, ?it/s]

TFMC models saved in ./demodir/models/TFMC/lowMT_VBFJet/tfmc_scan_do_1


### Calibrator

In [30]:
from ML.Calibration.Calibration import Calibration
from ML.TFMC.TFMC import TFMC 
calibrator_model_directory = os.path.join(model_dir, "Calibration")
os.makedirs(calibrator_model_directory, exist_ok=True)

calib = Calibration(
    yaml_config=os.path.join("configs", "config_submission.yaml"),
    selection=selection,
    small=True
)

calib.classifier = TFMC.load(calib.cfg['model_path'])
calib.model = calib.classifier

calib.training_data_dir = training_data_dir

def patched_train(self):
    print(f"Training: Load data for {self.selection}")

    import common.datasets_hephy as datasets_hephy
    self.loader = datasets_hephy.get_data_loader(
        data_directory=self.training_data_dir,
        selection=self.selection,
        n_split=self.n_split
    )

    all_prob = []
    all_weights = []
    all_labels = []

    from tqdm import tqdm
    for i, batch in enumerate(tqdm(self.loader, desc="Processing batches")):
        features, weights, labels = self.loader.split(batch)
        prob = self.model.predict(features)
        all_prob.append(prob)
        all_weights.append(weights)
        all_labels.append(labels)

    import numpy as np
    all_prob = np.concatenate(all_prob, axis=0)
    all_weights = np.concatenate(all_weights, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    all_prob /= all_prob.sum(axis=1, keepdims=True)

    from sklearn.isotonic import IsotonicRegression
    truth = (all_labels == 0).astype(float)
    self.iso_reg = IsotonicRegression(out_of_bounds='clip', y_min=1e-6, y_max=1.-1e-6).fit(all_prob[:, 0], truth, sample_weight=all_weights)
    self.data_for_plot = {"prob": all_prob, "weight": all_weights, "label": all_labels}

calib.train = patched_train.__get__(calib, Calibration)

print("Training Calibrator...")
calib.train()

calibrator_filename = os.path.join(calibrator_model_directory, f"calibrator_{selection}.pkl")
calib.save(calibrator_filename)
print(f"Saved calibrator to {calibrator_filename}")

plot_directory = os.path.join("demodir", "plots", "Calibration", selection)
os.makedirs(plot_directory, exist_ok=True)

calib.plot_calibration(os.path.join(plot_directory, 'calibrator_validation_calibration.png'))
calib.plot_IsotonicRegression(os.path.join(plot_directory, 'calibrator_validation_IsoReg.png'))

print(f"Saved calibration plots to {plot_directory}")



Training Calibrator...
Training: Load data for lowMT_VBFJet


Processing batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:19<00:00,  5.03it/s]


Saved calibrator to ./demodir/models/Calibration/calibrator_lowMT_VBFJet.pkl
Saved calibration plots to demodir/plots/Calibration/lowMT_VBFJet


### PNN

In [4]:
from ML.PNN.PNN import PNN

In [5]:
config_name = "ML.configs.pnn_quad_tes_jes_met"
# import the training parameters in config
config = importlib.import_module("%s"%( config_name))

for p in processes:
    subdirs= [arg for arg in [p, selection] if arg is not None]

    # Do we use ICP?
    if config.icp is not None:
        from ML.ICP.ICP import InclusiveCrosssectionParametrization
        icp_name = "ICP_" + "_".join(subdirs) + "_" + config.icp + "_light.pkl"
        icp = InclusiveCrosssectionParametrization.load(os.path.join(model_dir, "ICP", icp_name))
        config.icp_predictor = icp.get_predictor()
        print("We use this ICP:",icp_name)
        print(icp)
    
    # Do we use a Scaler?
    if config.use_scaler:
        from ML.Scaler.Scaler import Scaler
        scaler_name = "Scaler_"+"_".join(subdirs)+'.pkl'
        scaler = Scaler.load(os.path.join(model_dir, "Scaler", scaler_name))
        config.feature_means     = scaler.feature_means
        config.feature_variances = scaler.feature_variances
    
        print("We use this scaler:", scaler_name)
        print(scaler)
    
    # Where to store the training
    pnn_model_directory = os.path.join(model_dir, "PNN", *subdirs,  config_name)
    os.makedirs(pnn_model_directory, exist_ok=True)

# Initialize model
pnn = PNN(config)

# Initialize for training
pnn.load_training_data(datasets_hephy=datasets_hephy, training_data_dir=training_data_dir, process=p, selection=selection, n_split=100)

max_batch = 1 #if args.small else -1

# Training Loop
for epoch in range(0, config.n_epochs):

    # Manually evaluate and update the learning rate
    if hasattr(pnn, 'lr_schedule'):  # Ensure the schedule exists
        new_lr = pnn.lr_schedule(epoch)
        pnn.optimizer.learning_rate.assign(new_lr)  # Update the optimizer's learning rate
  
    # Print the current learning rate
    current_lr = tf.keras.backend.get_value(pnn.optimizer.learning_rate)  # Direct access
    print(f"Epoch {epoch}/{config.n_epochs} - Learning rate: {current_lr:.6f}")

    true_histograms, pred_histograms = pnn.train_one_epoch(max_batch=max_batch, accumulate_histograms=(epoch%1==0), rebin=1)
    pnn.save(pnn_model_directory, epoch)  # Save model and config after each epoch

    break


print(f"PNN models saved in {pnn_model_directory}")

2025-04-29 18:01:20.039955: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
I don't have the file for this choice: process: diboson values:(0.97, 1.0, 0.0)
Skipping missing file for process=diboson, values=(0.97, 1.0, 0.0)
I don't have the file for this choice: process: diboson values:(0.99, 0.99, 1.0)
Skipping missing file for process=diboson, values=(0.99, 0.99, 1.0)
I don't have the file for this choice: process: diboson values:(0.99, 0.99, 2.0)
Skipping missing file for process=diboson, values=(0.99, 0.99, 2.0)
I don't have the file for this choice: process: diboson values:(0.99, 1.0, 2.0)
Skipping missing file for process=diboson, values=(0.99, 1.0, 2.0)
I don't have the file for th

We use this ICP: ICP_htautau_lowMT_VBFJet_icp_quad_tes_jes_met_light.pkl
+6.0e-03*nu_tes +1.3e-02*nu_jes -2.6e-05*nu_met -1.0e-04*nu_tes*nu_tes -1.7e-04*nu_jes*nu_jes -9.2e-05*nu_met*nu_met +4.5e-05*nu_tes*nu_jes -2.3e-06*nu_tes*nu_met +2.1e-05*nu_jes*nu_met
We use this scaler: Scaler_htautau_lowMT_VBFJet.pkl
Scaler: selection lowMT_VBFJet process (htautau)
PRI_lep_pt: mean=47.784, variance=934.071
PRI_lep_eta: mean=0.000, variance=1.262
PRI_lep_phi: mean=-0.001, variance=3.288
PRI_had_pt: mean=64.702, variance=1609.351
PRI_had_eta: mean=0.001, variance=1.324
PRI_had_phi: mean=0.003, variance=3.285
PRI_jet_leading_pt: mean=127.257, variance=5997.741
PRI_jet_leading_eta: mean=-0.001, variance=3.695
PRI_jet_leading_phi: mean=-0.001, variance=3.269
PRI_jet_subleading_pt: mean=63.424, variance=1245.298
PRI_jet_subleading_eta: mean=0.000, variance=4.870
PRI_jet_subleading_phi: mean=-0.002, variance=3.229
PRI_n_jets: mean=2.893, variance=1.149
PRI_jet_all_pt: mean=225.778, variance=15460.502

Processing Batches: 0it [00:00, ?it/s]

PNN models saved in ./demodir/models/PNN/diboson/lowMT_VBFJet/ML.configs.pnn_quad_tes_jes_met


## Predict

For the predictions performed below, the fully trained model provided in `./models` are used.

In [4]:
from Workflow.Inference import Inference
from common.likelihoodFit import likelihoodFit
from common.intervalFinder import intervalFinder

Num GPUs Available:  0


### Load test dataset and config

A config file `./configs/config_submission.yaml` is used to control the path of the ML models. Please find the detailed information in the README of the GitHub repository.

In [5]:
def load_h5_to_test_set(h5_file_path):
    with h5py.File(h5_file_path, "r") as hf:
        full_data = hf["data"][:]  # (N, 30)
    test_data = pd.DataFrame(full_data[:, :28])  # (N, 28)
    test_weights = full_data[:, 28]  # (N,)
    test_set = {
        "data": test_data,
        "weights": test_weights
    }
    return test_set

def loadConfig(config_path):
    if config_path.endswith(".pkl"):
        use_yaml = False
    else:
        try:
            import yaml
            use_yaml = True
        except:
            import pickle
            use_yaml = False
            config_path = config_path.replace(".yaml", ".pkl")

    assert os.path.exists(config_path), "Config does not exist: {}".format(config_path)

    if use_yaml:
        with open(config_path) as f:
            cfg = yaml.safe_load(f)
    else:
        with open(config_path) as f:
            cfg = pickle.load(f, 'rb')

    for task in cfg["Tasks"]:
        for selection in cfg["Selections"]:
            for item in ["calibration", "icp_file", "model_path"]:
                if item in cfg[task][selection]:
                    cfg[task][selection][item] = cfg[task][selection][item]

    if "Poisson" in cfg:
        for sel in cfg["Poisson"].keys():
            if 'model_path' in cfg["Poisson"][sel]:
                cfg["Poisson"][sel]["model_path"] = cfg["Poisson"][sel]["model_path"]
            cfg["Poisson"][sel]["IC"] = cfg["Poisson"][sel]["IC"]
            for process in cfg["Poisson"][sel]["ICP"].keys():
                cfg["Poisson"][sel]["ICP"][process] = cfg["Poisson"][sel]["ICP"][process]
    
    cfg['tmp_path'] = f"data/tmp_data"

    return cfg

In [6]:
test_dataset_path = "/home/daohan/apps/Higgs/set_2.0_pseudo_exp_10.h5"
test_dataset = load_h5_to_test_set(test_dataset_path)

predict_config = "configs/config_submission.yaml"

### Pre-save information for training dataset

The prediction requires information from the training dataset for the inclusive cross section term. To reduce the runtime of the prediction steps, we pre-save the ML output for the training data, and save the cubic spline interpolation (CSI) for inclusive cross section and poisson terms. 

The pre-saved files are all data with the fully trained ML models are availible in `data/tmp_data`. Those files will be used later in the prediction.

In [8]:
cfg = loadConfig(predict_config)
cfg['training_data_dir'] = training_data_dir
cfg['selection'] = "lowMT_VBFJet"
cfg['Selections'] = ['lowMT_VBFJet'] 
save_key = list(cfg['Save'].keys())[0]
cfg['Save'][save_key]['dir'] = training_data_dir
cfg['tmp_path'] = "demodir/tmp_data"

if "Poisson" in cfg:
    for k in list(cfg["Poisson"].keys()):
        if cfg["Poisson"][k].get("preselection", None) != "lowMT_VBFJet":
            cfg["Poisson"][k]["ignore"] = True

infer = Inference(cfg, small=True, overwrite=True, toy_origin="config", toy_path=None, toy_from_memory=None)
infer.cfg["CSI"]["save"] = False
infer.save(restrict_csis=[])

for p in processes:
    cfg = loadConfig(predict_config)
    cfg['training_data_dir'] = training_data_dir
    cfg['selection'] = "lowMT_VBFJet"
    cfg['Selections'] = ['lowMT_VBFJet']
    save_key = list(cfg['Save'].keys())[0]
    cfg['Save'][save_key]['dir'] = training_data_dir
    cfg['tmp_path'] = "demodir/tmp_data"
    cfg['Toy_name'] = 'nominal'

    if "Poisson" in cfg:
        for k in list(cfg["Poisson"].keys()):
            if cfg["Poisson"][k].get("preselection", None) != "lowMT_VBFJet":
                cfg["Poisson"][k]["ignore"] = True

    infer = Inference(cfg, small=True, overwrite=True, toy_origin="config", toy_path=None, toy_from_memory=None)
    infer.cfg["CSI"]["save"] = True
    infer.save(restrict_csis=[p])



Warning! Temporary file demodir/tmp_data/nominal_lowMT_VBFJet.h5 exists. It will be overwritten.
Processing batches:   0%|                               | 0/100 [00:00<?, ?it/s]
Warning! Temporary file demodir/tmp_data/TrainingData_lowMT_VBFJet.h5 exists. It will be overwritten.
Processing batches:   0%|                               | 0/100 [00:00<?, ?it/s]
Warning! Temporary file demodir/tmp_data/nominal_lowMT_VBFJet.h5 exists. It will be overwritten.
Processing batches:   0%|                               | 0/100 [00:00<?, ?it/s]
Warning! Temporary file demodir/tmp_data/TrainingData_lowMT_VBFJet.h5 exists. It will be overwritten.
CSI lowMT_VBFJet htautau: 100%|███████████████████| 2/2 [00:09<00:00,  4.83s/it]
Warning! Temporary file demodir/tmp_data/nominal_lowMT_VBFJet.h5 exists. It will be overwritten.
Processing batches:   0%|                               | 0/100 [00:01<?, ?it/s]
Warning! Temporary file demodir/tmp_data/TrainingData_lowMT_VBFJet.h5 exists. It will be overwritten

### Construct the likelyhood function and extract the interval

In [9]:

offset = 0.0
inflate = 1.045
cfg['tmp_path'] = "data/tmp_data"

infer = Inference(cfg=cfg, small=False, overwrite=False, toy_origin="memory", toy_path=None, toy_from_memory=None)
infer.ignore_loading_check()

# Initialize inference object
infer.setToyFromMemory(test_dataset)
infer._dcr_cache = {}

# Define likelihood function
likelihood_function = lambda mu, nu_bkg, nu_tt, nu_diboson, nu_tes, nu_jes, nu_met: \
    infer.predict(mu=mu, nu_bkg=nu_bkg, nu_tt=nu_tt, nu_diboson=nu_diboson, \
    nu_tes=nu_tes, nu_jes=nu_jes, nu_met=nu_met, \
    asimov_mu=None, \
    asimov_nu_bkg=None, \
    asimov_nu_tt=None, \
    asimov_nu_diboson=None)

# Perform global fit
fit = likelihoodFit(likelihood_function)
fit.parameterBoundaries["mu"] = (0, None)
q_mle, parameters_mle, cov, limits = fit.fit(start_mu=1.0)

mu_mle = parameters_mle["mu"]
delta_mu = np.sqrt(cov["mu", "mu"])
p16 = mu_mle - delta_mu
p84 = mu_mle + delta_mu

# Now do NON-PROFILED scan
Npoints = 21
mumin = min(mu_mle - 3*delta_mu, mu_mle-0.5) # if delta_mu is too small, scan from mu-0.5 to mu+0.5
mumax = max(mu_mle + 3*delta_mu, mu_mle+0.5) # if delta_mu is too small, scan from mu-0.5 to mu+0.5

# Now go to MLE point and only evaluate mu
deltaQ = []
muPoints = [mumin+i*(mumax-mumin)/Npoints for i in range(Npoints)]

for i, mu in enumerate(muPoints):
    q = likelihood_function(mu=mu, nu_bkg=parameters_mle["nu_bkg"], nu_tt=parameters_mle["nu_tt"], nu_diboson=parameters_mle["nu_diboson"], nu_tes=parameters_mle["nu_tes"], nu_jes=parameters_mle["nu_jes"], nu_met=parameters_mle["nu_met"])
    deltaQ.append(q-q_mle)

# Interval finder interpolates and returns crossing points
# if a boundary is below best fit mu, it is the lower boundary, if above it is the upper

intFinder = intervalFinder(muPoints, deltaQ, 1.0)
boundaries = intFinder.getInterval()
for b in boundaries:
    if b < mu_mle:
        p16 = b
    if b > mu_mle:
        p84 = b

# inflate and offset

p16 = mu_mle - inflate*(mu_mle-p16) + offset
p84 = mu_mle + inflate*(p84-mu_mle) + offset
mu_mle = mu_mle + offset

# Check mu boundaries
if p16 < 0.1:
    p16 = 0.09
if p84 > 3.0:
    p84 = 3.01

delta_mu = (p84-p16)/2

results = {
            "mu_hat": mu_mle,
            "delta_mu_hat": delta_mu,
            "p16": p16,
            "p84": p84,
            "nu_bkg": parameters_mle["nu_bkg"],
            "nu_tt":  parameters_mle["nu_tt"],
            "nu_diboson": parameters_mle["nu_diboson"],
            "nu_tes": parameters_mle["nu_tes"],
            "nu_jes": parameters_mle["nu_jes"],
            "nu_met": parameters_mle["nu_met"],
        }

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = -2.919e+05                 │              Nfcn = 653              │
│ EDM = 1.7e-05 (Goal: 0.0001)     │            time = 3.4 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│     SOME parameters at limit     │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬────────────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬

### Results

In [10]:
print(results)

{'mu_hat': 32759.675368160875, 'delta_mu_hat': -16303.694743934533, 'p16': 32610.399487869065, 'p84': 3.01, 'nu_bkg': 2.6639183311004335, 'nu_tt': 9.999999657014184, 'nu_diboson': 3.9999990789319813, 'nu_tes': -8.14393911090881, 'nu_jes': -9.999999999677613, 'nu_met': 2.8488455491122375e-11}
